# Structure Upload

## PyXtal Structures
- first convert the cif files to a single csv with columns matching Structures schema

extract the following;
Field	Notes
structure_id	UUID or hash (we'll generate this)
source	"PyXtal"
source_id	Optional (leave blank or use file name)
generation_commit	Optional (can fill manually if known)
extinction_class	Can leave blank for now
composition	e.g., "SiO2"
lattice_a/b/c, alpha/beta/gamma	From CIF's cell parameters
wyckoff_positions	Optional — extract if available
atom_sites_json	JSON-encoded list of atom positions (from CIF)
creation_timestamp	Auto-generated in DB, not needed in CSV

### structures are generated by PyXtal and only exported to CIF with Pymatgen.
- structure = pyxtal(); structure.from_random(..., group=space_group, species=elements, numIons=ratios) → PyXtal randomly builds the 3D crystal for the requested space group.
- validate_structure(...) → you verify the generated structure’s symmetry using spglib.
- pmg_structure = pyxtal_structure.to_pymatgen(); CifWriter(pmg_structure).write_file(...) → you convert the PyXtal structure to a Pymatgen object and write the CIF.

the resulting CIFs are of PyXtal-generated (random) structures that passed our spglib check; Pymatgen is just the file writer here.

In [8]:
import os
import sys

# Get current working directory
cwd = os.getcwd()
print("Current working directory:", cwd)


Current working directory: /home/UserD/OrgA/crystal_database


### 1.1 Generating SG 2-230 cif files from PyXtal

In [12]:
import numpy as np
import random
import re
import json
import os
import uuid
import pyxtal
import spglib
from pyxtal import pyxtal
from pyxtal.symmetry import get_wyckoffs, choose_wyckoff
from pymatgen.io.cif import CifWriter

# Constants
VALID_SPACE_GROUPS = list(range(2, 231))
ELEMENTS = [
    'H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne',
    'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca',
    'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
    'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr',
    'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn',
    'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd',
    'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb',
    'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
    'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn', 'Fr', 'Ra', 'Ac', 'Th',
    'Pa', 'U'
]
OUTPUT_DIR = "/home/UserD/OrgA/generated_crystal_structures_pyxtal"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_valid_structure(space_group, max_attempts=1000):
    """Generates a valid structure with correct Wyckoff positions and compositions."""
    attempts = 0
    while attempts < max_attempts:
        try:
            elements, ratios = sample_valid_composition(space_group)
            structure = pyxtal()
            structure.from_random(dim=3, group=space_group, species=elements, numIons=ratios)

            if validate_structure(structure, space_group):
                return structure  # Success!
        except Exception as e:
            pass  # Ignore and retry

        attempts += 1

    raise ValueError(f"❌ Failed to generate valid structure for SG {space_group} after {max_attempts} attempts.")

def sample_valid_composition(space_group):
    """Ensures atom counts are compatible with available Wyckoff positions."""
    wyckoff_data = get_wyckoffs(space_group, organized=True)
    valid_counts = []

    for positions in wyckoff_data:
        total_atoms = sum(len(pos) for pos in positions)
        if total_atoms > 1:
            valid_counts.append(total_atoms)

    if not valid_counts:
        return ['C'], [2]

    num_atoms = random.choice(valid_counts)
    num_elements = np.random.randint(2, 4)
    elements = random.sample(ELEMENTS, num_elements)

    valid_ratios = [r for r in range(2, num_atoms + 1) if num_atoms % r == 0]
    ratios = [random.choice(valid_ratios) for _ in range(num_elements)]

    return elements, ratios

def validate_structure(structure, expected_space_group):
    """Checks if generated structure matches the requested space group."""
    atoms = structure.to_ase()
    lattice = atoms.get_cell()
    positions = atoms.get_scaled_positions()
    numbers = atoms.get_atomic_numbers()

    cell = (lattice, positions, numbers)
    detected_group = spglib.get_spacegroup(cell, symprec=1e-2)

    match = re.search(r'\((\d+)\)', detected_group)
    if match:
        detected_space_group = int(match.group(1))
        return expected_space_group == detected_space_group
    return False

def save_structure_as_cif(pyxtal_structure, space_group):
    """Converts pyxtal structure to CIF and saves to file. Returns path + metadata."""
    structure_id = str(uuid.uuid4())
    pmg_structure = pyxtal_structure.to_pymatgen()
    composition = pmg_structure.composition.reduced_formula

    # Build file name
    file_name = f"SG_{space_group}_{structure_id[:8]}.cif"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    # Save CIF
    try:
        writer = CifWriter(pmg_structure)
        writer.write_file(file_path)
        return {
            "structure_id": structure_id,
            "space_group": space_group,
            "file_path": file_path,
            "composition": composition,
            "success": True
        }
    except Exception as e:
        print(f"❌ Failed to write CIF for SG {space_group}: {e}")
        return {"success": False}

def generate_all_structures_with_cif():
    """Generates structures and saves them to CIF files."""
    valid_info = []
    failed_sgs = {}

    for sg in VALID_SPACE_GROUPS:
        try:
            structure = get_valid_structure(sg)
            result = save_structure_as_cif(structure, sg)

            if result["success"]:
                valid_info.append(result)
                print(f"✅ SG {sg} saved to {result['file_path']}")
            else:
                raise Exception("CIF write failed")

        except Exception as e:
            failed_sgs[sg] = str(e)
            print(f"🚨 Failed SG {sg}: {e}")

    print("\n📦 Generation Summary:")
    print(f"  ✅ Valid structures: {len(valid_info)}")
    print(f"  ❌ Failed SGs: {len(failed_sgs)}")
    print(f"  ❌ SGs with errors: {list(failed_sgs.keys())}")
    return valid_info, failed_sgs

# Run
structures_metadata, failed_sgs = generate_all_structures_with_cif()


✅ SG 2 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_2_3c1bb020.cif
✅ SG 3 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_3_e7b36731.cif
✅ SG 4 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_4_83b1af9d.cif
✅ SG 5 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_5_abdb664f.cif
✅ SG 6 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_6_799ac77a.cif
✅ SG 7 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_7_70fa05b4.cif
✅ SG 8 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_8_d7534bbb.cif
✅ SG 9 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_9_253aad5c.cif
✅ SG 10 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_10_abc6714a.cif
✅ SG 11 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_11_8766a5e6.cif
✅ SG 12 saved to /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_12_6cfc2ae2.cif
✅ SG 13 saved to /home/UserD/OrgA/gen

### 1.1a Generating SG 1 cif file from PyXtal 

In [ ]:
def generate_p1_structure():
    """Generates and saves a random P1 (SG 1) structure with full metadata and CIF."""
    structure = pyxtal()

    # Random composition
    num_elements = random.randint(1, 3)
    selected_elements = random.sample(ELEMENTS, num_elements)
    num_atoms = random.randint(2, 10)

    atom_distribution = [num_atoms // num_elements] * num_elements
    for i in range(num_atoms % num_elements):
        atom_distribution[i] += 1

    # Generate structure
    structure.from_random(dim=3, group=1, species=selected_elements, numIons=atom_distribution)

    # Generate UUID and filename
    structure_id = str(uuid.uuid4())
    file_name = f"SG_1_{structure_id[:8]}.cif"
    file_path = os.path.join(OUTPUT_DIR, file_name)

    # Convert to pymatgen and save as CIF
    try:
        pmg_structure = structure.to_pymatgen()
        writer = CifWriter(pmg_structure)
        writer.write_file(file_path)

        # Metadata for logging or export
        metadata = {
            "structure_id": structure_id,
            "space_group": 1,
            "composition": pmg_structure.composition.reduced_formula,
            "lattice_a": pmg_structure.lattice.a,
            "lattice_b": pmg_structure.lattice.b,
            "lattice_c": pmg_structure.lattice.c,
            "alpha": pmg_structure.lattice.alpha,
            "beta": pmg_structure.lattice.beta,
            "gamma": pmg_structure.lattice.gamma,
            "file_path": file_path
        }

        print("✅ Successfully generated and saved P1 structure:")
        for key, val in metadata.items():
            print(f"   - {key}: {val}")

        return metadata  # Can be added to a list later

    except Exception as e:
        print(f"❌ Failed to save P1 structure: {e}")
        return None

# Run it
p1_metadata = generate_p1_structure()

✅ Successfully generated and saved P1 structure:
   - structure_id: aa8fa976-59ab-4955-9d1e-9d634b0b7236
   - space_group: 1
   - composition: Y2At3N2
   - lattice_a: 6.479727483171309
   - lattice_b: 3.8395670702493576
   - lattice_c: 8.816749017808293
   - alpha: 121.61482924068845
   - beta: 94.84767468696474
   - gamma: 78.81833596946504
   - file_path: /home/UserD/OrgA/generated_crystal_structures_pyxtal/SG_1_aa8fa976.cif


### 1.2 Script to convert CIF Files to CSV
We use SpacegroupAnalyzer to detect and store the real space_group(ignores the SG in the filename),
- adds crystal_system, extinction_class (letter), extinction_group (numeric; optional mapping),
- sets source_id to the CIF filename (e.g., SG_113_8456e3d6.cif).
- extinction_class is pulled from the first letter of the H–M symbol (P, A, B, C, I, F, R).
- extinction_group is an ICSD numeric code; load your full mapping via EXTINCTION_GROUP_CSV, or extend the fallback dict.


load this CSV into a lookup dict. That way your script will automatically map each detected space group → extinction group + crystal system.
Loads your FINAL_SPG_ExtG_CrysS_Table.csv and looks up Extinction Group + Crystal System by space group number.

In [6]:
import os
import uuid
import csv
import json
import re
from typing import Dict, Tuple

from pymatgen.io.cif import CifParser
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# =========================
# CONFIG
# =========================
CIF_DIR = "/home/UserD/OrgA/generated_crystal_structures_pyxtal"
OUTPUT_CSV = "pyxtal_structures.csv"

SOURCE = "PyXtal"
GENERATION_COMMIT = "unknown"

# Path to your mapping file
EXTINCTION_GROUP_CSV = "FINAL_SPG_ExtG_CrysS_Table.csv"


# =========================
# HELPERS
# =========================
def load_extinction_map() -> Dict[int, Tuple[int, str]]:
    """
    Load SG → (extinction group, crystal system) mapping.
    Returns {space_group: (extinction_group, crystal_system)}
    """
    mapping = {}
    with open(EXTINCTION_GROUP_CSV, "r", newline="") as f:
        reader = csv.DictReader(f, delimiter=",")
        for row in reader:
            try:
                sg = int(row["Space Group"])
                eg = int(row["Extinction Group"])
                cs = row["Crystal System"].capitalize()
                mapping[sg] = (eg, cs)
            except Exception as e:
                print(f"⚠️ Skipping row {row}: {e}")
    return mapping


def extinction_class_from_hm(hm: str) -> str:
    """Return centering letter (P, A, B, C, I, F, R) from H–M symbol."""
    if not hm:
        return "P"
    s = hm.strip().replace(" ", "")
    first = s[0].upper()
    return first if first in ("P", "A", "B", "C", "I", "F", "R") else "P"


# =========================
# MAIN EXTRACTOR
# =========================
def extract_structure_data(cif_path, sg_map: Dict[int, Tuple[int, str]]):
    try:
        parser = CifParser(cif_path)
        structure = parser.get_structures()[0]

        analyzer = SpacegroupAnalyzer(structure, symprec=1e-2)
        sg_symbol = analyzer.get_space_group_symbol()
        sg_number = analyzer.get_space_group_number()

        wyckoffs = analyzer.get_symmetry_dataset().get("wyckoffs", [])

        atom_data = []
        for site in structure.sites:
            atom_data.append({
                "element": site.specie.symbol,
                "frac_coords": [float(x) for x in site.frac_coords],
            })

        extinction_class = extinction_class_from_hm(sg_symbol)
        extinction_group, crystal_system = sg_map.get(sg_number, (None, "Unknown"))

        return {
            "structure_id": str(uuid.uuid4()),
            "source": SOURCE,
            "source_id": os.path.basename(cif_path),  # filename
            "generation_commit": GENERATION_COMMIT,
            "extinction_class": extinction_class,
            "extinction_group": extinction_group,
            "crystal_system": crystal_system,
            "space_group": sg_number,
            "composition": structure.composition.reduced_formula,
            "lattice_a": float(structure.lattice.a),
            "lattice_b": float(structure.lattice.b),
            "lattice_c": float(structure.lattice.c),
            "alpha": float(structure.lattice.alpha),
            "beta": float(structure.lattice.beta),
            "gamma": float(structure.lattice.gamma),
            "wyckoff_positions": ",".join(wyckoffs),
            "atom_sites_json": json.dumps(atom_data),
        }

    except Exception as e:
        print(f"❌ Failed to parse {cif_path}: {e}")
        return None


def write_csv_from_cifs():
    sg_map = load_extinction_map()
    records = []

    for fname in os.listdir(CIF_DIR):
        if not fname.endswith(".cif"):
            continue
        cif_path = os.path.join(CIF_DIR, fname)
        data = extract_structure_data(cif_path, sg_map)
        if data:
            records.append(data)

    fieldnames = [
        "structure_id",
        "source",
        "source_id",
        "generation_commit",
        "extinction_class",
        "extinction_group",
        "crystal_system",
        "space_group",
        "composition",
        "lattice_a",
        "lattice_b",
        "lattice_c",
        "alpha",
        "beta",
        "gamma",
        "wyckoff_positions",
        "atom_sites_json",
        "creation_timestamp",
    ]

    with open(OUTPUT_CSV, mode="w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            r["creation_timestamp"] = None
            writer.writerow(r)

    print(f"✅ CSV written: {OUTPUT_CSV} with {len(records)} rows.")


# Run
if __name__ == "__main__":
    write_csv_from_cifs()


/tmp/ipykernel_773778/935703373.py:61: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structure = parser.get_structures()[0]


✅ CSV written: pyxtal_structures.csv with 544 rows.


### 1.3 Inserting PyXtal structures into Structures Table

#### 1.3a Loading csv cif files into Structures Table

In [9]:
import sqlite3
import csv

# Connect to your SQLite database
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

# Load the CSV
with open("pyxtal_structures.csv", newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        cursor.execute("""
            INSERT INTO Structures (
                structure_id,
                source,
                source_id,
                generation_commit,
                extinction_class,
                extinction_group,
                crystal_system,
                space_group,
                composition,
                lattice_a,
                lattice_b,
                lattice_c,
                alpha,
                beta,
                gamma,
                wyckoff_positions,
                atom_sites_json
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,?,?)
        """, (
            row["structure_id"],
            row["source"],
            row["source_id"],
            row["generation_commit"],
            row["extinction_class"],
            int(row["extinction_group"]),
            row["crystal_system"],
            int(row["space_group"]),
            row["composition"],
            float(row["lattice_a"]),
            float(row["lattice_b"]),
            float(row["lattice_c"]),
            float(row["alpha"]),
            float(row["beta"]),
            float(row["gamma"]),
            row["wyckoff_positions"],
            row["atom_sites_json"]
        ))

# Commit and close
conn.commit()
conn.close()

print("All structure rows (with space group) loaded into the database successfully!")


All structure rows (with space group) loaded into the database successfully!


#### Scripting to Verify the PyXtal upload in the Structures Table

In [10]:
import sqlite3

# Connect to the database
conn = sqlite3.connect("crystal_xrd.db")
cursor = conn.cursor()

# Query some rows from the Structures table
cursor.execute("SELECT * FROM Structures LIMIT 10;")
rows = cursor.fetchall()

# Print each row
for row in rows:
    print("\n--- Structure ---")
    for col, val in zip([d[0] for d in cursor.description], row):
        print(f"{col}: {val}")

# Count total rows
cursor.execute("SELECT COUNT(*) FROM Structures;")
total = cursor.fetchone()[0]
print(f"\n Total rows in Structures table: {total}")

# Close connection
conn.close()



--- Structure ---
structure_id: 42c91502-ed0e-4f79-92da-f71335eba43d
space_group: 120
source: PyXtal
source_id: SG_120_feab4e5f.cif
generation_commit: unknown
extinction_class: I
extinction_group: 66
crystal_system: Tetragonal
composition: B2N
lattice_a: 6.073939079507657
lattice_b: 6.073939079507656
lattice_c: 6.073939079507655
alpha: 106.6511578799445
beta: 106.6511578799445
gamma: 115.27175779627322
wyckoff_positions: e,e,e,e,a,a,c,c,h,h,h,h
atom_sites_json: [{"element": "B", "frac_coords": [0.8502558100000001, 0.6497441900000002, 0.0]}, {"element": "B", "frac_coords": [0.6497441899999999, 0.8502558099999997, 1.0]}, {"element": "B", "frac_coords": [0.35025580999999995, 0.35025580999999995, 0.20051161999999986]}, {"element": "B", "frac_coords": [0.14974419000000005, 0.14974419, 0.7994883800000001]}, {"element": "B", "frac_coords": [0.25, 0.25, 0.0]}, {"element": "B", "frac_coords": [0.75, 0.75, 0.0]}, {"element": "B", "frac_coords": [0.25, 0.75, 0.5]}, {"element": "B", "frac_coords"

## 2. ICSD Structures

In [ ]:
import os
import sys

# Get current working directory
cwd = os.getcwd()
print("Current working directory:", cwd)
'''/home/UserD/OrgA/Findit_CIFS'''

Current working directory: /home/UserD/OrgA/crystal_database


In [12]:
import sqlite3

conn = sqlite3.connect("crystal_xrd.db")
cur = conn.cursor()

# Add new columns if they don't already exist
cur.execute("PRAGMA table_info(Structures);")
cols = [row[1] for row in cur.fetchall()]

if "publication_year" not in cols:
    cur.execute("ALTER TABLE Structures ADD COLUMN publication_year INTEGER;")

if "ICSD_creation_date" not in cols:
    cur.execute("ALTER TABLE Structures ADD COLUMN ICSD_creation_date TEXT;")

conn.commit()
conn.close()

print("Columns ensured in Structures table.")


Columns ensured in Structures table.


In [24]:
import os
import uuid
import csv
import json
import time
import math
import sqlite3
from typing import Dict, Tuple, Optional

from pymatgen.io.cif import CifParser
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# =========================
# CONFIG
# =========================
DB_PATH = "crystal_xrd.db"
ICSD_CIF_DIR = "/home/UserD/OrgA/FindIt_CIFs"   # your ICSD folder
SPG_MAP_CSV = "FINAL_SPG_ExtG_CrysS_Table.csv"  # columns: Space Group,Extinction Group,Crystal System

SOURCE = "ICSD"
GENERATION_COMMIT = "unknown"
SYM_TOL = 1e-2

# Behavior
SKIP_IF_SOURCE_ID_EXISTS = True   # skip duplicates by source_id (filename)
BATCH_COMMIT_EVERY = 1000
PRINT_EVERY = 500                 # print progress every N parsed files
OCC_TOL = 1.1                     # occupancy tolerance for CIF parser

# =========================
# DB SCHEMA (uses your existing columns)
# =========================
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS Structures (
    structure_id TEXT PRIMARY KEY,
    source TEXT,
    source_id TEXT,
    generation_commit TEXT,
    extinction_class TEXT,
    space_group INTEGER,
    composition TEXT,
    lattice_a REAL,
    lattice_b REAL,
    lattice_c REAL,
    alpha REAL,
    beta REAL,
    gamma REAL,
    wyckoff_positions TEXT,
    atom_sites_json TEXT,
    extinction_group INTEGER,
    crystal_system TEXT,
    publication_year INTEGER,
    ICSD_creation_date TEXT,
    creation_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

def ensure_schema(conn: sqlite3.Connection):
    cur = conn.cursor()
    cur.execute(CREATE_TABLE_SQL)
    cur.execute("CREATE INDEX IF NOT EXISTS idx_structures_source_id ON Structures(source_id);")
    conn.commit()

def set_fast_pragmas(conn: sqlite3.Connection):
    cur = conn.cursor()
    cur.execute("PRAGMA journal_mode=WAL;")
    cur.execute("PRAGMA synchronous=OFF;")
    cur.execute("PRAGMA temp_store=MEMORY;")
    # OK if ignored by OS:
    cur.execute("PRAGMA mmap_size=30000000000;")
    conn.commit()

INSERT_SQL = """
INSERT INTO Structures (
    structure_id, source, source_id, generation_commit,
    extinction_class, extinction_group, crystal_system, space_group,
    composition, lattice_a, lattice_b, lattice_c, alpha, beta, gamma,
    wyckoff_positions, atom_sites_json, publication_year, ICSD_creation_date
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

# =========================
# MAPPING + HELPERS
# =========================
def load_spg_map(csv_path: str) -> Dict[int, Tuple[int, str]]:
    mapping = {}
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                sg = int(row["Space Group"])
                eg = int(row["Extinction Group"])
                cs = row["Crystal System"].strip().capitalize()
                mapping[sg] = (eg, cs)
            except Exception as e:
                print(f"Skipping map row {row}: {e}")
    return mapping

def extinction_class_from_hm(hm: str) -> str:
    if not hm:
        return "P"
    s = hm.strip().replace(" ", "")
    first = s[0].upper()
    return first if first in ("P","A","B","C","I","F","R") else "P"

def source_id_exists(conn: sqlite3.Connection, source_id: str) -> bool:
    cur = conn.cursor()
    cur.execute("SELECT 1 FROM Structures WHERE source_id = ? LIMIT 1;", (source_id,))
    return cur.fetchone() is not None

def _safe_first(val):
    if isinstance(val, list):
        return val[0] if val else None
    return val

def _norm_date(d: Optional[str]) -> Optional[str]:
    if not d:
        return None
    d = str(d).strip().replace(".", "/").replace("-", "/")
    parts = d.split("/")
    if len(parts) == 3:
        y, m, day = parts
        return f"{int(y):04d}-{int(m):02d}-{int(float(day)):02d}"
    if len(parts) == 1 and len(parts[0]) == 4:
        return f"{int(parts[0]):04d}-01-01"
    return d

def iter_cif_paths(root: str):
    # recursive, in case your folder has subdirs
    for r, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(".cif"):
                yield os.path.join(r, f)

def site_records(site):
    """
    Return a list of atom records for one site.
    Handles ordered (single specie) and disordered (multiple species with occupancies).
    """
    coords = [float(x) for x in site.frac_coords]
    if getattr(site, "is_ordered", True):
        # ordered site
        try:
            el = site.specie.symbol  # works when ordered
            return [{"element": el, "occupancy": 1.0, "frac_coords": coords}]
        except AttributeError:
            # fallback to parsed string
            return [{"element": site.species_string, "occupancy": 1.0, "frac_coords": coords}]
    else:
        # disordered site
        recs = []
        for sp, occ in site.species.items():
            recs.append({
                "element": sp.symbol,       # strip oxidation state
                "occupancy": float(occ),
                "frac_coords": coords,
            })
        return recs

# =========================
# CIF → ROW
# =========================
def cif_to_row(cif_path: str, spg_map: Dict[int, Tuple[int, str]]):
    # Robust parser: tolerate partial occupancies; keep conventional cell
    parser = CifParser(cif_path, occupancy_tolerance=OCC_TOL)
    structures = parser.get_structures(primitive=False)
    if not structures:
        raise ValueError("No structure parsed")
    structure = structures[0]
    # Remove oxidation states to help symmetry detection
    try:
        structure = structure.remove_oxidation_states()
    except Exception:
        pass

    sga = SpacegroupAnalyzer(structure, symprec=SYM_TOL)
    sg_symbol = sga.get_space_group_symbol()
    sg_number = sga.get_space_group_number()

    ds = sga.get_symmetry_dataset()
    wyckoffs = ds.get("wyckoffs", []) if ds else []

    atom_data = []
    for site in structure.sites:
        atom_data.extend(site_records(site))

    extinction_group, crystal_system = spg_map.get(sg_number, (None, "Unknown"))
    extinction_class = extinction_class_from_hm(sg_symbol)

    # ICSD metadata
    cif_dict = parser.as_dict()
    first_block = next(iter(cif_dict.values()))
    pub_year = None
    for k, v in first_block.items():
        if "citation_year" in k.lower():
            try:
                pub_year = int(_safe_first(v))
            except Exception:
                pass
            break
    if pub_year is None:
        py = first_block.get("_publ_year")
        try:
            pub_year = int(_safe_first(py)) if py is not None else None
        except Exception:
            pass

    icsd_creation_date = _norm_date(_safe_first(first_block.get("_audit_creation_date")))
    source_id = os.path.basename(cif_path)  # per your request

    return (
        str(uuid.uuid4()),
        SOURCE,
        source_id,
        GENERATION_COMMIT,
        extinction_class,
        extinction_group,
        crystal_system,
        int(sg_number),
        structure.composition.reduced_formula,
        float(structure.lattice.a),
        float(structure.lattice.b),
        float(structure.lattice.c),
        float(structure.lattice.alpha),
        float(structure.lattice.beta),
        float(structure.lattice.gamma),
        ",".join(wyckoffs),
        json.dumps(atom_data),
        pub_year,
        icsd_creation_date,
    )

# =========================
# MAIN with progress & ETA
# =========================
def human_time(seconds: float) -> str:
    if math.isinf(seconds) or seconds < 0:
        return "unknown"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h: return f"{h}h {m}m {s}s"
    if m: return f"{m}m {s}s"
    return f"{s}s"

def load_all_icsd():
    if not os.path.isdir(ICSD_CIF_DIR):
        raise FileNotFoundError(f"CIF directory not found: {ICSD_CIF_DIR}")

    spg_map = load_spg_map(SPG_MAP_CSV)

    conn = sqlite3.connect(DB_PATH)
    set_fast_pragmas(conn)
    ensure_schema(conn)
    cur = conn.cursor()

    # collect list first to compute ETA
    paths = list(iter_cif_paths(ICSD_CIF_DIR))
    total = len(paths)
    print(f"Found {total} CIF files in {ICSD_CIF_DIR}")

    inserted = skipped = failed = 0
    batch = 0
    start = time.time()
    last_print = start

    for i, path in enumerate(paths, 1):
        fname = os.path.basename(path)
        try:
            if SKIP_IF_SOURCE_ID_EXISTS and source_id_exists(conn, fname):
                skipped += 1
                continue

            row = cif_to_row(path, spg_map)
            cur.execute(INSERT_SQL, row)
            inserted += 1
            batch += 1

            if batch >= BATCH_COMMIT_EVERY:
                conn.commit()
                batch = 0

        except Exception as e:
            failed += 1
            # keep going; just report filename + short error
            print(f"{fname}: {e}")

        # progress / ETA print
        if (i % PRINT_EVERY == 0) or (i == total):
            elapsed = time.time() - start
            rate = i / elapsed if elapsed > 0 else float("inf")
            remaining = (total - i) / rate if rate > 0 else float("inf")
            print(f"[{i}/{total}] inserted={inserted}, skipped={skipped}, failed={failed} | "
                  f"elapsed={human_time(elapsed)}, eta={human_time(remaining)}")

    conn.commit()
    conn.close()
    print(f"Done. Inserted: {inserted}, Skipped: {skipped}, Failed: {failed} | "
          f"Total time: {human_time(time.time() - start)}")

if __name__ == "__main__":
    load_all_icsd()


Found 181362 CIF files in /home/UserD/OrgA/FindIt_CIFs


/tmp/ipykernel_773778/2830242646.py:166: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structures = parser.get_structures(primitive=False)


icsd_614197.cif: zip() argument 2 is longer than argument 1
icsd_074677.cif: Invalid CIF file with no structures!
icsd_247110.cif: too close distance between atoms
icsd_194339.cif: too close distance between atoms
icsd_192085.cif: too close distance between atoms
icsd_083733.cif: zip() argument 2 is longer than argument 1
icsd_102496.cif: zip() argument 2 is longer than argument 1
icsd_055449.cif: zip() argument 2 is longer than argument 1
icsd_633967.cif: zip() argument 2 is longer than argument 1
icsd_647050.cif: zip() argument 2 is longer than argument 1
icsd_618993.cif: zip() argument 2 is longer than argument 1
icsd_039724.cif: zip() argument 2 is longer than argument 1
icsd_652642.cif: zip() argument 2 is longer than argument 1
icsd_605818.cif: zip() argument 2 is longer than argument 1
icsd_625406.cif: len(items)=23 is not a multiple of n=2
icsd_153378.cif: Invalid CIF file with no structures!
icsd_102671.cif: zip() argument 2 is longer than argument 1
icsd_419689.cif: zip() arg

/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1312: UserWarning: Some occupancies ([1.00019]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.1
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1347: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
Some occupancies ([1.00019]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 1.1
  return self.parse_structures(*args, **kwargs)


icsd_640488.cif: zip() argument 2 is longer than argument 1
icsd_646013.cif: zip() argument 2 is longer than argument 1
icsd_066048.cif: Invalid CIF file with no structures!


...

icsd_625491.cif: zip() argument 2 is longer than argument 1


/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1312: UserWarning: Incorrect stoichiometry:
  CIF={'Cr': 1.0, 'Li': 4.0, 'O': 6.0, 'Sb': 1.0}
  PMG={'Li': 16.080000000000005, 'Cr': 3.920000000000001, 'Sb': 4.0, 'O': 24.0}
  ratios={'Li': 4.020000000000001, 'Cr': 3.920000000000001, 'O': 4.0, 'Sb': 4.0}
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


icsd_624712.cif: zip() argument 2 is longer than argument 1
icsd_607502.cif: zip() argument 2 is longer than argument 1
icsd_059794.cif: len(items)=26 is not a multiple of n=3
icsd_186397.cif: zip() argument 2 is longer than argument 1
icsd_636616.cif: zip() argument 2 is longer than argument 1


/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1312: UserWarning: Incorrect stoichiometry:
  CIF={'Fe': 0.19, 'Mg': 1.81, 'O': 4.0, 'Si': 1.0}
  PMG={'Mg': 7.248, 'Fe': 0.7519999999999999, 'Si': 4.0, 'O': 16.0}
  ratios={'O': 4.0, 'Mg': 4.0044198895027625, 'Si': 4.0, 'Fe': 3.9578947368421047}
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):


icsd_625907.cif: zip() argument 2 is longer than argument 1
icsd_391175.cif: len(items)=29 is not a multiple of n=3
icsd_636372.cif: zip() argument 2 is longer than argument 1
icsd_601754.cif: len(items)=23 is not a multiple of n=2
[181362/181362] inserted=174190, skipped=0, failed=7172 | elapsed=1h 36m 57s, eta=0s
Done. Inserted: 174190, Skipped: 0, Failed: 7172 | Total time: 1h 36m 57s


In [ ]:
#check who has the database open !lsof crystal_xrd.db

#### To force close and truncate wal file

In [14]:
import sqlite3

DB_PATH = "crystal_xrd.db"
conn = sqlite3.connect(DB_PATH, timeout=60.0)
cur = conn.cursor()

# Merge WAL into the main DB; TRUNCATE will also shrink/remove the WAL file if possible.
cur.execute("PRAGMA wal_checkpoint(FULL);")
cur.execute("PRAGMA wal_checkpoint(TRUNCATE);")

# Optional: VACUUM reclaims free space in the main DB (takes time; needs exclusive access)
cur.execute("VACUUM;")

conn.commit()
conn.close()


### Troubleshooting and Getting the files that didn't load

In [6]:
import os, sqlite3, csv

ICSD_CIF_DIR = "/home/UserD/OrgA/FindIt_CIFs"
DB_PATH = "crystal_xrd.db"
OUT_CSV = "icsd_missing_files.csv"

# 1. Files in folder
all_files = {f for f in os.listdir(ICSD_CIF_DIR) if f.lower().endswith(".cif")}

# 2. Files already loaded
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT source_id FROM Structures WHERE source='ICSD';")
db_files = {row[0] for row in cur.fetchall()}
conn.close()

# 3. Find missing
missing = all_files - db_files
print(f"Total CIFs in folder: {len(all_files)}")
print(f"Loaded into DB: {len(db_files)}")
print(f"Missing: {len(missing)}")

# 4. Save missing to CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename"])
    for fname in sorted(missing):
        writer.writerow([fname])

print(f"Missing file list saved to {OUT_CSV}")


Total CIFs in folder: 181362
Loaded into DB: 174190
Missing: 7172
Missing file list saved to icsd_missing_files.csv


Which filenames were actually inserted

In [7]:
import sqlite3
conn = sqlite3.connect("crystal_xrd.db")
cur = conn.cursor()
cur.execute("SELECT source_id FROM Structures WHERE source='ICSD';")
inserted = {r[0] for r in cur.fetchall()}
conn.close()
print(len(inserted))


174190


Which CIF filenames exist on disk (recursive)

In [8]:
import os
root = "/home/UserD/OrgA/FindIt_CIFs"
on_disk = set()
for r,_,files in os.walk(root):
    for f in files:
        if f.lower().endswith(".cif"):
            on_disk.add(f)   # filename only, like your loader’s source_id
print(len(on_disk))


181362


Missing (on disk but not inserted)

In [9]:
missing = on_disk - inserted
print("Missing:", len(missing))


Missing: 7172


Duplicates on disk (same filename appears >1 time in the tree)

In [10]:
from collections import defaultdict
dups = defaultdict(list)
for r,_,files in os.walk(root):
    for f in files:
        if f.lower().endswith(".cif"):
            dups[f].append(os.path.join(r,f))
dup_names = {name:paths for name,paths in dups.items() if len(paths)>1}
print("Duplicate filenames:", len(dup_names))
# Example peek:
for k,v in list(dup_names.items())[:5]:
    print(k, "->", len(v), "copies")


Duplicate filenames: 0


In [12]:
import os, csv, json, sqlite3, uuid
from typing import Dict, Tuple, Optional
from pymatgen.io.cif import CifParser
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

DB_PATH = "crystal_xrd.db"
ICSD_CIF_DIR = "/home/UserD/OrgA/FindIt_CIFs"
MISSING_CSV = "icsd_missing_files.csv"
SPG_MAP_CSV = "FINAL_SPG_ExtG_CrysS_Table.csv"

SYM_TOL = 1e-2
SOURCE = "ICSD"
GENERATION_COMMIT = "unknown"

def load_spg_map(csv_path: str) -> Dict[int, Tuple[int, str]]:
    mapping = {}
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            sg = int(row["Space Group"])
            eg = int(row["Extinction Group"])
            cs = row["Crystal System"].strip().capitalize()
            mapping[sg] = (eg, cs)
    return mapping

def extinction_class_from_hm(hm: str) -> str:
    if not hm: return "P"
    first = hm.strip().replace(" ", "")[0].upper()
    return first if first in ("P","A","B","C","I","F","R") else "P"

def site_records(site):
    coords = [float(x) for x in site.frac_coords]
    if getattr(site, "is_ordered", True):
        try:
            el = site.specie.symbol
            return [{"element": el, "occupancy": 1.0, "frac_coords": coords}]
        except AttributeError:
            return [{"element": site.species_string, "occupancy": 1.0, "frac_coords": coords}]
    else:
        recs = []
        for sp, occ in site.species.items():
            recs.append({"element": sp.symbol, "occupancy": float(occ), "frac_coords": coords})
        return recs

def parse_with_tolerance(cif_path: str, occ_tol: float):
    parser = CifParser(cif_path, occupancy_tolerance=occ_tol)
    structs = parser.get_structures(primitive=False)
    if not structs:
        return None, None, None
    s = structs[0]
    try:
        s = s.remove_oxidation_states()
    except Exception:
        pass
    return s, parser, structs

def build_row(path: str, spg_map):
    # try two tolerances
    for occ_tol in (2.5, 10.0):
        try:
            structure, parser, _ = parse_with_tolerance(path, occ_tol)
            if structure is None:
                continue
            sga = SpacegroupAnalyzer(structure, symprec=SYM_TOL)
            sg_symbol = sga.get_space_group_symbol()
            sg_number = sga.get_space_group_number()
            ds = sga.get_symmetry_dataset()
            wyckoffs = ds.get("wyckoffs", []) if ds else []

            atom_data = []
            for site in structure.sites:
                atom_data.extend(site_records(site))

            extinction_group, crystal_system = spg_map.get(sg_number, (None, "Unknown"))
            extinction_class = extinction_class_from_hm(sg_symbol)

            cif_dict = parser.as_dict()
            first_block = next(iter(cif_dict.values()))
            pub_year = None
            for k, v in first_block.items():
                if "citation_year" in k.lower():
                    try:
                        pub_year = int(v[0] if isinstance(v, list) else v)
                    except Exception:
                        pass
                    break
            if pub_year is None:
                py = first_block.get("_publ_year")
                try:
                    pub_year = int(py[0] if isinstance(py, list) else py) if py is not None else None
                except Exception:
                    pass

            # date normalization (simple)
            def _norm_date(d):
                if not d: return None
                d = str(d).strip().replace(".", "/").replace("-", "/")
                parts = d.split("/")
                if len(parts) == 3:
                    y,m,day = parts
                    return f"{int(y):04d}-{int(m):02d}-{int(float(day)):02d}"
                if len(parts) == 1 and len(parts[0]) == 4:
                    return f"{int(parts[0]):04d}-01-01"
                return d

            icsd_creation_date = _norm_date(first_block.get("_audit_creation_date"))
            source_id = os.path.basename(path)

            row = (
                str(uuid.uuid4()), SOURCE, source_id, GENERATION_COMMIT,
                extinction_class, extinction_group, crystal_system, int(sg_number),
                structure.composition.reduced_formula,
                float(structure.lattice.a), float(structure.lattice.b), float(structure.lattice.c),
                float(structure.lattice.alpha), float(structure.lattice.beta), float(structure.lattice.gamma),
                ",".join(wyckoffs), json.dumps(atom_data), pub_year, icsd_creation_date
            )
            return row
        except Exception:
            continue
    return None

def main():
    spg_map = load_spg_map(SPG_MAP_CSV)
    # load missing names
    with open(MISSING_CSV, "r", encoding="utf-8") as f:
        names = [r[0] for r in csv.reader(f)][1:]

    # map filename -> full path (recursive)
    pathmap = {}
    for r,_,files in os.walk(ICSD_CIF_DIR):
        for f in files:
            if f.lower().endswith(".cif"):
                pathmap[f] = os.path.join(r,f)

    conn = sqlite3.connect(DB_PATH, timeout=60.0)
    cur = conn.cursor()

    INSERT_SQL = """
    INSERT INTO Structures (
        structure_id, source, source_id, generation_commit,
        extinction_class, extinction_group, crystal_system, space_group,
        composition, lattice_a, lattice_b, lattice_c, alpha, beta, gamma,
        wyckoff_positions, atom_sites_json, publication_year, ICSD_creation_date
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
    """

    recovered = failed = 0
    with open("icsd_missing_recovery_log.csv", "w", newline="", encoding="utf-8") as lf:
        lw = csv.writer(lf); lw.writerow(["filename","status","note"])
        for name in names:
            path = pathmap.get(name)
            if not path:
                lw.writerow([name, "failed", "file not found on disk"])
                failed += 1
                continue
            row = build_row(path, spg_map)
            if row is None:
                lw.writerow([name, "failed", "still not parseable with high occupancy_tolerance"])
                failed += 1
                continue
            try:
                cur.execute(INSERT_SQL, row)
                recovered += 1
                lw.writerow([name, "inserted", "lenient parse"])
            except Exception as e:
                lw.writerow([name, "failed", str(e)[:300]])
                failed += 1
        conn.commit()
    conn.close()
    print(f"Recovered: {recovered}, Still failed: {failed}")

if __name__ == "__main__":
    main()


/tmp/ipykernel_1367672/2288144493.py:47: FutureWarning: get_structures is deprecated; use parse_structures in pymatgen.io.cif instead.
The only difference is that primitive defaults to False in the new parse_structures method.So parse_structures(primitive=True) is equivalent to the old behavior of get_structures().
  structs = parser.get_structures(primitive=False)
/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1312: UserWarning: Some occupancies ([1.27]) sum to > 1! If they are within the occupancy_tolerance, they will be rescaled. The current occupancy_tolerance is set to: 2.5
  if struct := self._get_structure(data, primitive, symmetrized, check_occu=check_occu):
/home/UserD/anaconda3/envs/pyxtal_env/lib/python3.11/site-packages/pymatgen/io/cif.py:1312: UserWarning: Incorrect stoichiometry:
  CIF={'Al': 22.0, 'O': 34.0, 'Tl': 2.0}
  PMG={'Al': 22.0, 'Tl': 1.8903968503937008, 'O': 33.574803149606296}
  ratios={'O': 0.9874942102825381, 'Tl': 0.94

Recovered: 409, Still failed: 6763


In [15]:
import os, sqlite3, csv

ICSD_CIF_DIR = "/home/UserD/OrgA/FindIt_CIFs"
DB_PATH = "crystal_xrd.db"
OUT_CSV = "icsd_missing_files2.csv"

# 1. Files in folder
all_files = {f for f in os.listdir(ICSD_CIF_DIR) if f.lower().endswith(".cif")}

# 2. Files already loaded
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT source_id FROM Structures WHERE source='ICSD';")
db_files = {row[0] for row in cur.fetchall()}
conn.close()

# 3. Find missing
missing = all_files - db_files
print(f"Total CIFs in folder: {len(all_files)}")
print(f"Loaded into DB: {len(db_files)}")
print(f"Missing: {len(missing)}")

# 4. Save missing to CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename"])
    for fname in sorted(missing):
        writer.writerow([fname])

print(f"Missing file list saved to {OUT_CSV}")

import sqlite3
conn = sqlite3.connect("crystal_xrd.db")
cur = conn.cursor()
cur.execute("SELECT source_id FROM Structures WHERE source='ICSD';")
inserted = {r[0] for r in cur.fetchall()}
conn.close()
print(len(inserted))

missing = on_disk - inserted
print("Missing:", len(missing))


Total CIFs in folder: 181362
Loaded into DB: 174599
Missing: 6763
Missing file list saved to icsd_missing_files2.csv
174599
Missing: 6763


Some ICSD CIFs failed to load because of inconsistencies in how the files were written. In many cases, the atomic site occupancies summed to values greater than 1, which exceeded the parsing tolerance and caused Pymatgen to reject the structure. Other files had mismatched stoichiometry between the reported chemical formula and the atomic positions, or contained implicit atoms (like hydrogens) that weren’t explicitly defined. A smaller fraction were malformed or incomplete CIFs that couldn’t be converted into a valid structure at all.